In [7]:
import os
import re
from collections import Counter, defaultdict

import camelot
import pandas as pd
from pypdf import PdfReader


# =========================================================
# 1. 路徑設定
# =========================================================

BASE_FOLDER = r"C:\test2"

PDF_FILE = os.path.join(
    BASE_FOLDER,
    "112年報_20251125.pdf"
)

OUTPUT_FOLDER = os.path.join(
    BASE_FOLDER,
    "篩選結果"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)


# =========================================================
# 2. 自動尋找癌症頁碼 Excel
# =========================================================

def find_page_excel():

    candidates = []

    for filename in os.listdir(BASE_FOLDER):

        if filename.startswith("~$"):
            continue

        if not filename.lower().endswith(
            (".xlsx", ".xls")
        ):
            continue

        if (
            "癌" in filename
            and "頁" in filename
        ):
            candidates.append(filename)

    if not candidates:

        raise FileNotFoundError(
            "找不到癌症頁碼 Excel"
        )

    return os.path.join(
        BASE_FOLDER,
        candidates[0]
    )


PAGE_EXCEL = find_page_excel()


# =========================================================
# 3. 基本文字清理
# =========================================================

def clean_text(value):

    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def compact_text(value):

    return re.sub(
        r"\s+",
        "",
        clean_text(value)
    )


def safe_filename(value):

    return re.sub(
        r'[\\/:*?"<>|]',
        "_",
        clean_text(value)
    )


# =========================================================
# 4. Camelot 常見文字修正
# =========================================================

def fix_common_text(text):

    text = clean_text(text)

    replacements = {
        "TNM 診斷組 合": "TNM 診斷組合",
        "TNM 診斷 組合": "TNM 診斷組合",
        "TNM診斷組 合": "TNM診斷組合",

        "臨 床期 別": "臨床期別",
        "臨床 期別": "臨床期別",

        "病 理期 別": "病理期別",
        "病理 期別": "病理期別",

        "整 併期 別": "整併期別",
        "整併 期別": "整併期別",
        "整 併 期 別": "整併期別",
        "整併期 別": "整併期別",

        "總 個 案數": "總個案數",
        "總 個案數": "總個案數",

        "其 他/ 不詳": "其他/不詳",
        "其 他 / 不詳": "其他/不詳",
        "其他 / 不詳": "其他/不詳",
        "其他／不詳": "其他/不詳",

        "S tage": "Stage",
        "ST AGE": "Stage",

        "II I": "III",
        "I II": "III",
        "I V": "IV",

        "II I A": "IIIA",
        "II I B": "IIIB",

        "I V A": "IVA",
        "I V B": "IVB",
        "I V C": "IVC",
    }

    for old, new in replacements.items():

        text = text.replace(
            old,
            new
        )

    return text.strip()


def fix_dataframe_text(df):

    return df.map(
        lambda value:
        fix_common_text(
            str(value)
        )
    )


# =========================================================
# 5. 頁碼整理
# =========================================================

def normalize_pages(value):

    if pd.isna(value):
        return ""

    text = str(value).strip()

    replacements = {
        "、": ",",
        "，": ",",
        "～": "-",
        "~": "-",
        "－": "-",
        "–": "-",
        "—": "-"
    }

    for old, new in replacements.items():

        text = text.replace(
            old,
            new
        )

    text = text.replace(
        "頁",
        ""
    )

    text = re.sub(
        r"\s+",
        "",
        text
    )

    text = re.sub(
        r"\.0(?=,|-|$)",
        "",
        text
    )

    return text


def expand_page_range(page_range):

    pages = []

    for part in page_range.split(","):

        part = part.strip()

        if not part:
            continue

        if "-" in part:

            start, end = part.split(
                "-",
                1
            )

            pages.extend(
                range(
                    int(start),
                    int(end) + 1
                )
            )

        else:

            pages.append(
                int(part)
            )

    return pages


# =========================================================
# 6. Camelot 表格位置
# =========================================================

def get_table_top(table):

    return table._bbox[3]


# =========================================================
# 7. Camelot DataFrame 清理
# =========================================================

def clean_dataframe(df):

    df = df.copy()

    df = df.map(
        lambda value:
        re.sub(
            r"\s+",
            " ",
            str(value)
        ).strip()
        if pd.notna(value)
        else ""
    )

    # 只刪除完全空白列
    # 絕對不能刪除空白欄
    empty_rows = df.apply(
        lambda row:
        row
        .astype(str)
        .str.strip()
        .eq("")
        .all(),
        axis=1
    )

    df = df.loc[
        ~empty_rows
    ]

    return df.reset_index(
        drop=True
    )


# =========================================================
# 8. Camelot 擷取
# =========================================================

def extract_pdf_tables(page_range):

    pages = expand_page_range(
        page_range
    )

    page_dataframes = []

    for page_number in pages:

        print(
            f"  讀取第 {page_number} 頁"
        )

        tables = camelot.read_pdf(
            PDF_FILE,
            pages=str(page_number),
            flavor="stream",
            split_text=False,
            row_tol=7,
            column_tol=3,
            edge_tol=500
        )

        if len(tables) == 0:

            print(
                f"  第 {page_number} 頁沒有表格"
            )

            continue

        # 同一頁由上往下
        sorted_tables = sorted(
            tables,
            key=lambda table:
            -get_table_top(table)
        )

        for table_number, table in enumerate(
            sorted_tables,
            start=1
        ):

            df = clean_dataframe(
                table.df.copy()
            )

            df = fix_dataframe_text(
                df
            )

            if df.empty:
                continue

            print(
                f"    表格 {table_number}："
                f"{df.shape[0]} 列 × "
                f"{df.shape[1]} 欄"
            )

            page_dataframes.append(
                {
                    "page": page_number,
                    "top": get_table_top(
                        table
                    ),
                    "df": df
                }
            )

    if not page_dataframes:

        return None

    # 頁碼 + 頁面上下位置
    page_dataframes = sorted(
        page_dataframes,
        key=lambda item: (
            item["page"],
            -item["top"]
        )
    )

    # 最大欄數
    max_columns = max(
        item["df"].shape[1]
        for item in page_dataframes
    )

    print(
        f"  最大欄位數：{max_columns}"
    )

    aligned = []

    for item in page_dataframes:

        df = item["df"].copy()

        # 保留所有原本欄位
        df.columns = range(
            df.shape[1]
        )

        # 不足只在右邊補
        df = df.reindex(
            columns=range(
                max_columns
            ),
            fill_value=""
        )

        aligned.append(
            df
        )

    return pd.concat(
        aligned,
        ignore_index=True
    )


# =========================================================
# 9. PDF 年度
# =========================================================

PDF_READER = PdfReader(
    PDF_FILE
)


def get_year_from_pdf(page_range):

    years = []

    for page_number in expand_page_range(
        page_range
    ):

        page_index = page_number - 1

        if (
            page_index < 0
            or page_index >= len(
                PDF_READER.pages
            )
        ):
            continue

        try:

            text = (
                PDF_READER.pages[
                    page_index
                ].extract_text()
                or ""
            )

        except Exception:

            continue

        found = re.findall(
            r"(?:民國\s*)?(1\d{2})\s*年",
            text
        )

        for year in found:

            if year not in years:
                years.append(year)

    if years:

        return str(
            max(
                int(year)
                for year in years
            )
        )

    # 從 PDF 檔名補
    match = re.search(
        r"(\d{3})年報",
        os.path.basename(
            PDF_FILE
        )
    )

    if match:

        return match.group(1)

    return ""


# =========================================================
# 10. DataFrame 整列文字
# =========================================================

def get_row_text(
    df,
    row_index
):

    texts = []

    for value in df.iloc[
        row_index
    ]:

        text = clean_text(
            value
        )

        if text:

            texts.append(
                text
            )

    return " ".join(
        texts
    )


def get_row_compact_text(
    df,
    row_index
):

    return compact_text(
        get_row_text(
            df,
            row_index
        )
    )


# =========================================================
# 11. 找某個標題
# =========================================================

def find_row_containing(
    df,
    keyword,
    start=0,
    end=None
):

    if end is None:

        end = len(df)

    target = compact_text(
        keyword
    )

    for row_index in range(
        start,
        end
    ):

        row_text = (
            get_row_compact_text(
                df,
                row_index
            )
        )

        if target in row_text:

            return row_index

    return None


# =========================================================
# 12. 找第一個非空文字
# =========================================================

def get_first_nonempty_text(
    df,
    row_index,
    max_columns=6
):

    max_columns = min(
        max_columns,
        df.shape[1]
    )

    for col in range(
        max_columns
    ):

        text = clean_text(
            df.iloc[
                row_index,
                col
            ]
        )

        if text:

            return text

    return ""


# =========================================================
# 13. 數值 token 處理
# =========================================================

def convert_number(token):

    if token in {
        "-",
        "－",
        "–",
        "—"
    }:

        return 0

    token = (
        str(token)
        .replace(",", "")
        .strip()
    )

    try:

        number = float(
            token
        )

        if number.is_integer():

            return int(
                number
            )

        return number

    except ValueError:

        return 0


# =========================================================
# 14. 擷取統計 token
# =========================================================

def extract_data_tokens_from_text(text):
    """
    非常重要：

    只抓真正獨立的統計數字。

    不會抓：
    T1
    N0
    M0

    裡面的數字。
    """

    return re.findall(
        r"(?<![A-Za-z0-9])"
        r"(?:\d+(?:\.\d+)?|[-－–—])"
        r"(?![A-Za-z0-9])",
        text
    )


def get_row_data_tokens(
    df,
    row_index
):

    tokens = []

    for value in df.iloc[
        row_index
    ]:

        text = clean_text(
            value
        )

        if not text:
            continue

        tokens.extend(
            extract_data_tokens_from_text(
                text
            )
        )

    return tokens


# =========================================================
# 15. 核心：抓五個 N
# =========================================================

def get_five_n_values(
    df,
    row_index
):
    """
    這就是之前成功的核心方式。

    不管前面有：

    Stage
    T
    N
    M
    PSA
    Grade

    都不管。

    直接取最右側十個：

    N % N % N % N % N %

    再取：

    第 0
    第 2
    第 4
    第 6
    第 8

    就是：

    合計
    男性
    女性
    醫學中心
    非醫學中心
    """

    tokens = get_row_data_tokens(
        df,
        row_index
    )

    if len(tokens) < 10:

        raise ValueError(
            f"第 {row_index} 列統計資料不足："
            f"{tokens}"
        )

    last_ten = tokens[
        -10:
    ]

    return [
        convert_number(
            last_ten[0]
        ),
        convert_number(
            last_ten[2]
        ),
        convert_number(
            last_ten[4]
        ),
        convert_number(
            last_ten[6]
        ),
        convert_number(
            last_ten[8]
        )
    ]


# =========================================================
# 16. 年齡標準化
# =========================================================

def normalize_age(value):

    text = clean_text(
        value
    )

    compact = compact_text(
        text
    )

    # 00-04歲
    match = re.fullmatch(
        r"(\d{1,3})[-~～–—](\d{1,3})歲?",
        compact
    )

    if match:

        start_age = int(
            match.group(1)
        )

        end_age = int(
            match.group(2)
        )

        return (
            f"{start_age:02d}-"
            f"{end_age:02d} 歲"
        )

    # 85歲以上
    match = re.fullmatch(
        r"(\d{1,3})歲?以上",
        compact
    )

    if match:

        return (
            f"{int(match.group(1))} 歲以上"
        )

    return None


# =========================================================
# 17. 年齡擷取
# =========================================================

def extract_age(
    df,
    cancer_name,
    year
):

    output_columns = [
        "年度",
        "癌別",
        "年齡",
        "合計",
        "男性",
        "女性",
        "醫學中心",
        "非醫學中心"
    ]

    # =====================================================
    # 找年齡
    # =====================================================

    age_start = find_row_containing(
        df,
        "年齡"
    )

    if age_start is None:

        print(
            "    找不到年齡"
        )

        return pd.DataFrame(
            columns=output_columns
        )


    print(
        f"    年齡開始：第 {age_start} 列"
    )


    # =====================================================
    # 年齡結束
    # =====================================================

    age_end = len(df)

    for i in range(
        age_start + 1,
        len(df)
    ):

        text = get_row_compact_text(
            df,
            i
        )

        if (
            "臨床期別" in text
            or "病理期別" in text
            or "整併期別" in text
            or "TNM診斷組合" in text
        ):

            age_end = i

            break


    # =====================================================
    # 年齡格式
    # =====================================================

    age_pattern = re.compile(
        r"^\d{1,3}\s*[-~～–—]\s*"
        r"\d{1,3}\s*歲?$"
        r"|^\d{1,3}\s*歲以上$"
    )


    records = []


    for i in range(
        age_start + 1,
        age_end
    ):

        item = get_first_nonempty_text(
            df,
            i,
            max_columns=6
        )

        item = clean_text(
            item
        )


        if not age_pattern.fullmatch(
            item
        ):

            continue


        age = normalize_age(
            item
        )


        if age is None:

            continue


        try:

            values = get_five_n_values(
                df,
                i
            )

        except ValueError:

            continue


        records.append(
            {
                "年度": year,
                "癌別": cancer_name,
                "年齡": age,
                "合計": values[0],
                "男性": values[1],
                "女性": values[2],
                "醫學中心": values[3],
                "非醫學中心": values[4]
            }
        )


    result = pd.DataFrame(
        records,
        columns=output_columns
    )


    # =====================================================
    # 跨頁重複年齡：去重
    # =====================================================

    if not result.empty:

        result = (
            result
            .drop_duplicates(
                subset=[
                    "年度",
                    "癌別",
                    "年齡"
                ],
                keep="first"
            )
            .reset_index(
                drop=True
            )
        )


    return result


# =========================================================
# 18. TNM 左側文字判斷
# =========================================================

def is_pure_statistical_cell(
    text
):

    text = clean_text(
        text
    )

    if not text:

        return False


    normalized = text.replace(
        ",",
        ""
    )


    return bool(
        re.fullmatch(
            r"(?:\d+(?:\.\d+)?|[-－–—])"
            r"(?:\s+(?:\d+(?:\.\d+)?|[-－–—]))*",
            normalized
        )
    )


# =========================================================
# 19. 取得 TNM 左側資料
# =========================================================

def get_tnm_left_cells(
    df,
    row_index
):
    """
    找到右側統計資料之前的文字。

    例如：

    IIIA T3a
    N0
    M0
    """

    left_cells = []


    for value in df.iloc[
        row_index
    ]:

        text = clean_text(
            value
        )


        if not text:

            continue


        # 遇到純統計數值就停止
        if is_pure_statistical_cell(
            text
        ):

            break


        left_cells.append(
            text
        )


    return left_cells


# =========================================================
# 20. Stage 名稱
# =========================================================

STAGE_NAMES = [
    "其他/不詳",

    "IIIA",
    "IIIB",
    "IIIC",

    "IVA",
    "IVB",
    "IVC",

    "IIA",
    "IIB",
    "IIC",

    "III",
    "IV",
    "II",
    "I",
    "0",
]


# =========================================================
# 21. Stage + T 分離
# =========================================================

def split_stage_and_t(
    text
):

    text = clean_text(
        text
    )

    if not text:

        return "", ""


    if (
        "不詳" in compact_text(
            text
        )
    ):

        return "不詳", ""


    for stage in STAGE_NAMES:

        if text == stage:

            return stage, ""


        pattern = (
            rf"^{re.escape(stage)}"
            r"\s+(.+)$"
        )


        match = re.match(
            pattern,
            text,
            flags=re.IGNORECASE
        )


        if match:

            return (
                stage,
                match.group(1).strip()
            )


    return "", text


# =========================================================
# 22. 拆 TNM 左側欄位
# =========================================================

def parse_tnm_left_columns(
    df,
    row_index
):

    cells = get_tnm_left_cells(
        df,
        row_index
    )


    if not cells:

        return [
            "",
            "",
            "",
            "",
            "",
            ""
        ]


    stage, t_value = (
        split_stage_and_t(
            cells[0]
        )
    )


    remaining = cells[1:]


    n_value = ""
    m_value = ""
    psa_value = ""
    grade_value = ""


    if len(remaining) >= 1:

        n_value = remaining[0]


    if len(remaining) >= 2:

        m_value = remaining[1]


    if len(remaining) >= 3:

        psa_value = remaining[2]


    if len(remaining) >= 4:

        grade_value = " ".join(
            remaining[3:]
        )


    return [
        stage,
        t_value,
        n_value,
        m_value,
        psa_value,
        grade_value
    ]


# =========================================================
# 23. 主整併期別判斷
# =========================================================

def normalize_main_stage(stage):
    """
    我們最後工作表只要：

    0
    I
    II
    III
    IV
    不詳

    不要：

    IIA
    IIIA
    IVA
    IVB
    等細分類。
    """

    text = compact_text(
        stage
    ).upper()


    if not text:

        return None


    if "不詳" in text:

        return "不詳"


    mapping = {
        "0": "0",
        "I": "I",
        "II": "II",
        "III": "III",
        "IV": "IV"
    }


    return mapping.get(
        text
    )


# =========================================================
# 24. 判斷是否為 TNM 特殊表
# =========================================================

def is_tnm_table(df):

    for i in range(
        len(df)
    ):

        text = get_row_compact_text(
            df,
            i
        ).upper()


        if "TNM診斷組合" in text:

            return True


        if (
            "STAGE" in text
            and "T" in text
            and "N" in text
            and "M" in text
        ):

            return True


    return False


# =========================================================
# 25. 特殊 TNM 整併期別
# =========================================================

def extract_tnm_stage(
    df,
    cancer_name,
    year
):
    """
    完全沿用你之前成功的：

    parse_tnm_left_columns
    +
    get_five_n_values

    方式。

    最後只保留主 Stage：
    0 / I / II / III / IV / 不詳
    """

    output_columns = [
        "年度",
        "癌別",
        "整併期別",
        "合計",
        "男性",
        "女性",
        "醫學中心",
        "非醫學中心"
    ]


    merged_start = find_row_containing(
        df,
        "整併期別"
    )


    if merged_start is None:

        merged_start = find_row_containing(
            df,
            "TNM診斷組合"
        )


    if merged_start is None:

        print(
            "    找不到 TNM 整併期別"
        )

        return pd.DataFrame(
            columns=output_columns
        )


    print(
        f"    TNM 整併期別開始："
        f"第 {merged_start} 列"
    )


    merged_end = len(df)


    # =====================================================
    # 找下一個大區塊
    # =====================================================

    for i in range(
        merged_start + 1,
        len(df)
    ):

        text = get_row_compact_text(
            df,
            i
        )


        if (
            "一致性" in text
            or "治療" in text
            or "存活" in text
            or "組織型態" in text
            or "組織形態" in text
        ):

            merged_end = i

            break


    records = []


    # =====================================================
    # 逐列分析
    # =====================================================

    for i in range(
        merged_start + 1,
        merged_end
    ):

        row_text = get_row_compact_text(
            df,
            i
        )


        if not row_text:

            continue


        # 表頭不要
        if (
            "STAGE" in row_text.upper()
            and "N" in row_text.upper()
            and "M" in row_text.upper()
        ):

            continue


        if "整併期別" in row_text:

            continue


        # =================================================
        # 先讀右側 N
        # =================================================

        try:

            n_values = get_five_n_values(
                df,
                i
            )

        except ValueError:

            continue


        # =================================================
        # 再解析左側 TNM
        # =================================================

        left_values = (
            parse_tnm_left_columns(
                df,
                i
            )
        )


        if not any(
            left_values
        ):

            continue


        raw_stage = left_values[0]


        main_stage = normalize_main_stage(
            raw_stage
        )


        # IIA / IIIA / IVA 等細項全部不要
        if main_stage is None:

            continue


        # =================================================
        # 主 Stage 必須沒有 T/N/M 細項
        #
        # 例：
        # III      350 ...
        #
        # 可以。
        #
        # IIIA T3a N0 M0
        #
        # 不可以。
        # =================================================

        if main_stage != "不詳":

            t_value = clean_text(
                left_values[1]
            )

            n_value = clean_text(
                left_values[2]
            )

            m_value = clean_text(
                left_values[3]
            )


            if (
                t_value
                or n_value
                or m_value
            ):

                continue


        records.append(
            {
                "年度": year,
                "癌別": cancer_name,
                "整併期別": main_stage,
                "合計": n_values[0],
                "男性": n_values[1],
                "女性": n_values[2],
                "醫學中心": n_values[3],
                "非醫學中心": n_values[4]
            }
        )


    result = pd.DataFrame(
        records,
        columns=output_columns
    )


    if not result.empty:

        result = (
            result
            .drop_duplicates(
                subset=[
                    "年度",
                    "癌別",
                    "整併期別"
                ],
                keep="first"
            )
            .reset_index(
                drop=True
            )
        )


    return result


# =========================================================
# 26. 一般癌症整併期別
# =========================================================

def normalize_normal_stage(value):

    text = compact_text(
        value
    ).upper()


    if not text:

        return None


    if "不詳" in text:

        return "不詳"


    text = text.replace(
        "期",
        ""
    )


    mapping = {
        "0": "0",
        "I": "I",
        "II": "II",
        "III": "III",
        "IV": "IV",
        "IVA": "IVA",
        "IVB": "IVB",
        "IVC": "IVC",
        "IVM0": "IV M0",
        "IVM1": "IV M1"
    }


    return mapping.get(
        text
    )


def extract_normal_stage(
    df,
    cancer_name,
    year
):

    output_columns = [
        "年度",
        "癌別",
        "整併期別",
        "合計",
        "男性",
        "女性",
        "醫學中心",
        "非醫學中心"
    ]


    start = find_row_containing(
        df,
        "整併期別"
    )


    if start is None:

        return pd.DataFrame(
            columns=output_columns
        )


    records = []


    for i in range(
        start + 1,
        len(df)
    ):

        row_text = get_row_compact_text(
            df,
            i
        )


        # 後續大區塊停止
        upper = row_text.upper()


        if records:

            if (
                upper.startswith("STAGE")
                or upper.startswith("GRADE")
                or upper.startswith("SA(")
            ):

                break


        item = get_first_nonempty_text(
            df,
            i,
            max_columns=5
        )


        stage = normalize_normal_stage(
            item
        )


        if stage is None:

            continue


        try:

            values = get_five_n_values(
                df,
                i
            )

        except ValueError:

            continue


        records.append(
            {
                "年度": year,
                "癌別": cancer_name,
                "整併期別": stage,
                "合計": values[0],
                "男性": values[1],
                "女性": values[2],
                "醫學中心": values[3],
                "非醫學中心": values[4]
            }
        )


    result = pd.DataFrame(
        records,
        columns=output_columns
    )


    if not result.empty:

        result = (
            result
            .drop_duplicates(
                subset=[
                    "年度",
                    "癌別",
                    "整併期別"
                ],
                keep="first"
            )
            .reset_index(
                drop=True
            )
        )


    return result


# =========================================================
# 27. 整併期別自動切換
# =========================================================

def extract_stage(
    df,
    cancer_name,
    year
):

    if is_tnm_table(
        df
    ):

        print(
            "    偵測到 TNM 特殊表格"
        )

        return extract_tnm_stage(
            df,
            cancer_name,
            year
        )


    print(
        "    使用一般整併期別"
    )


    return extract_normal_stage(
        df,
        cancer_name,
        year
    )


# =========================================================
# 28. 所有空值補 0
# =========================================================

def fill_numeric_zero(df):

    numeric_columns = [
        "合計",
        "男性",
        "女性",
        "醫學中心",
        "非醫學中心"
    ]


    if df.empty:

        return df


    for column in numeric_columns:

        if column in df.columns:

            df[column] = (
                df[column]
                .fillna(0)
            )


    return df


# =========================================================
# 29. Excel 欄寬
# =========================================================

def adjust_excel_width(
    writer,
    sheet_name
):

    worksheet = writer.book[
        sheet_name
    ]


    for cells in worksheet.columns:

        max_length = 0

        letter = (
            cells[0].column_letter
        )


        for cell in cells:

            value = (
                ""
                if cell.value is None
                else str(cell.value)
            )


            max_length = max(
                max_length,
                len(value)
            )


        worksheet.column_dimensions[
            letter
        ].width = min(
            max_length + 3,
            25
        )


# =========================================================
# 30. 讀癌症頁碼 Excel
# =========================================================

page_df = pd.read_excel(
    PAGE_EXCEL
)


print()

print(
    "癌症頁碼 Excel 欄位：",
    list(page_df.columns)
)


# =========================================================
# 31. 找癌別與頁碼欄位
# =========================================================

cancer_column = None
page_column = None


for column in page_df.columns:

    text = compact_text(
        column
    )


    if cancer_column is None:

        if (
            "癌別" in text
            or "癌症" in text
            or "名稱" in text
        ):

            cancer_column = column


    if page_column is None:

        if (
            "頁碼" in text
            or "頁" in text
        ):

            page_column = column


if cancer_column is None:

    cancer_column = (
        page_df.columns[0]
    )


if page_column is None:

    page_column = (
        page_df.columns[1]
    )


print(
    "癌別欄：",
    cancer_column
)

print(
    "頁碼欄：",
    page_column
)


# =========================================================
# 32. 計算癌別出現次數
# =========================================================

all_cancers = []


for _, row in page_df.iterrows():

    cancer_name = clean_text(
        row[cancer_column]
    )

    if cancer_name:

        all_cancers.append(
            cancer_name
        )


total_count = Counter(
    all_cancers
)


current_count = defaultdict(
    int
)


# =========================================================
# 33. 成功 / 失敗紀錄
# =========================================================

success_list = []
failed_list = []


# =========================================================
# 34. 批次處理
# =========================================================

for excel_index, row in page_df.iterrows():

    cancer_name = clean_text(
        row[cancer_column]
    )


    page_range = normalize_pages(
        row[page_column]
    )


    if not cancer_name:
        continue


    if not page_range:
        continue


    current_count[
        cancer_name
    ] += 1


    occurrence = current_count[
        cancer_name
    ]


    # =====================================================
    # 同癌別兩筆就分開
    # =====================================================

    if total_count[
        cancer_name
    ] > 1:

        output_filename = (
            f"{safe_filename(cancer_name)}"
            f"_{occurrence}.xlsx"
        )

    else:

        output_filename = (
            f"{safe_filename(cancer_name)}"
            ".xlsx"
        )


    output_file = os.path.join(
        OUTPUT_FOLDER,
        output_filename
    )


    print()

    print(
        "=" * 70
    )

    print(
        f"癌別：{cancer_name}"
    )

    print(
        f"PDF 頁碼：{page_range}"
    )


    if total_count[
        cancer_name
    ] > 1:

        print(
            f"同癌別第 {occurrence} 筆"
        )


    print(
        "=" * 70
    )


    try:

        # =================================================
        # A. 年度
        # =================================================

        year = get_year_from_pdf(
            page_range
        )


        print(
            f"  年度：{year}"
        )


        # =================================================
        # B. Camelot
        # =================================================

        merged_df = extract_pdf_tables(
            page_range
        )


        if merged_df is None:

            raise ValueError(
                "Camelot 沒有可用資料"
            )


        print(
            f"  合併後："
            f"{merged_df.shape[0]} 列 × "
            f"{merged_df.shape[1]} 欄"
        )


        # =================================================
        # C. 年齡
        # =================================================

        age_df = extract_age(
            merged_df,
            cancer_name,
            year
        )


        # =================================================
        # D. 整併期別
        # =================================================

        stage_df = extract_stage(
            merged_df,
            cancer_name,
            year
        )


        # =================================================
        # E. 空白補 0
        # =================================================

        age_df = fill_numeric_zero(
            age_df
        )


        stage_df = fill_numeric_zero(
            stage_df
        )


        # =================================================
        # F. 最後再次去重
        # =================================================

        if not age_df.empty:

            age_df = (
                age_df
                .drop_duplicates(
                    subset=[
                        "年度",
                        "癌別",
                        "年齡"
                    ],
                    keep="first"
                )
                .reset_index(
                    drop=True
                )
            )


        if not stage_df.empty:

            stage_df = (
                stage_df
                .drop_duplicates(
                    subset=[
                        "年度",
                        "癌別",
                        "整併期別"
                    ],
                    keep="first"
                )
                .reset_index(
                    drop=True
                )
            )


        print(
            f"  年齡："
            f"{len(age_df)} 筆"
        )


        print(
            f"  整併期別："
            f"{len(stage_df)} 筆"
        )


        # =================================================
        # G. 輸出 Excel
        # =================================================

        with pd.ExcelWriter(
            output_file,
            engine="openpyxl"
        ) as writer:


            age_df.to_excel(
                writer,
                sheet_name="年齡",
                index=False
            )


            stage_df.to_excel(
                writer,
                sheet_name="整併期別",
                index=False
            )


            adjust_excel_width(
                writer,
                "年齡"
            )


            adjust_excel_width(
                writer,
                "整併期別"
            )


        print(
            f"  完成：{output_filename}"
        )


        success_list.append(
            {
                "癌別": cancer_name,
                "第幾次": occurrence,
                "PDF頁碼": page_range,
                "年度": year,
                "年齡筆數": len(
                    age_df
                ),
                "整併期別筆數": len(
                    stage_df
                ),
                "輸出檔案": output_filename
            }
        )


    except Exception as error:

        print(
            f"  發生錯誤：{error}"
        )


        failed_list.append(
            {
                "癌別": cancer_name,
                "第幾次": occurrence,
                "PDF頁碼": page_range,
                "錯誤原因": str(
                    error
                )
            }
        )


# =========================================================
# 35. 處理結果
# =========================================================

report_file = os.path.join(
    OUTPUT_FOLDER,
    "處理結果.xlsx"
)


success_df = pd.DataFrame(
    success_list
)

failed_df = pd.DataFrame(
    failed_list
)


with pd.ExcelWriter(
    report_file,
    engine="openpyxl"
) as writer:


    success_df.to_excel(
        writer,
        sheet_name="成功",
        index=False
    )


    failed_df.to_excel(
        writer,
        sheet_name="失敗",
        index=False
    )


    adjust_excel_width(
        writer,
        "成功"
    )


    adjust_excel_width(
        writer,
        "失敗"
    )


# =========================================================
# 36. 完成
# =========================================================

print()

print(
    "=" * 70
)

print(
    "全部處理完成"
)

print(
    "=" * 70
)

print(
    f"成功：{len(success_list)}"
)

print(
    f"失敗：{len(failed_list)}"
)

print()

print(
    "輸出位置："
)

print(
    OUTPUT_FOLDER
)


癌症頁碼 Excel 欄位： ['癌別', '頁碼']
癌別欄： 癌別
頁碼欄： 頁碼

癌別：口腔癌
PDF 頁碼：562-563
  年度：112
  讀取第 562 頁
    表格 1：40 列 × 11 欄
    表格 2：37 列 × 11 欄
  讀取第 563 頁
    表格 1：41 列 × 12 欄
    表格 2：10 列 × 7 欄
  最大欄位數：12
  合併後：128 列 × 12 欄
    年齡開始：第 5 列
    使用一般整併期別
  年齡：18 筆
  整併期別：7 筆
  完成：口腔癌.xlsx

癌別：口咽癌
PDF 頁碼：568-569
同癌別第 1 筆
  年度：112
  讀取第 568 頁
    表格 1：36 列 × 12 欄
  讀取第 569 頁
    表格 1：30 列 × 12 欄
    表格 2：10 列 × 4 欄
  最大欄位數：12
  合併後：76 列 × 12 欄
    年齡開始：第 5 列
    使用一般整併期別
  年齡：18 筆
  整併期別：5 筆
  完成：口咽癌_1.xlsx

癌別：口咽癌
PDF 頁碼：570-571
同癌別第 2 筆
  年度：112
  讀取第 570 頁
    表格 1：39 列 × 11 欄
    表格 2：37 列 × 11 欄
  讀取第 571 頁
    表格 1：41 列 × 12 欄
    表格 2：10 列 × 7 欄
  最大欄位數：12
  合併後：127 列 × 12 欄
    年齡開始：第 4 列
    使用一般整併期別
  年齡：18 筆
  整併期別：7 筆
  完成：口咽癌_2.xlsx

癌別：下咽癌
PDF 頁碼：578-579
  年度：112
  讀取第 578 頁
    表格 1：39 列 × 11 欄
  讀取第 579 頁
    表格 1：41 列 × 12 欄
    表格 2：10 列 × 7 欄
  最大欄位數：12
  合併後：90 列 × 12 欄
    年齡開始：第 4 列
    使用一般整併期別
  年齡：18 筆
  整併期別：7 筆
  完成：下咽癌.xlsx

癌別：喉癌
PDF 頁碼：584-585
  年度：112
  讀取第 584 頁
    表格

## Excel統整

In [9]:
import os
import pandas as pd

from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter


# =========================================================
# 1. 資料夾設定
# =========================================================

input_folder = r"C:\test2\篩選結果"

output_file = os.path.join(
    input_folder,
    "全部癌症統整.xlsx"
)


# =========================================================
# 2. 儲存所有癌症資料
# =========================================================

age_dataframes = []
stage_dataframes = []


# =========================================================
# 3. 取得資料夾裡所有 Excel
# =========================================================

excel_files = []

for filename in os.listdir(
    input_folder
):

    # 只讀 xlsx
    if not filename.lower().endswith(
        ".xlsx"
    ):
        continue

    # 不讀 Excel 暫存檔
    if filename.startswith(
        "~$"
    ):
        continue

    # 不讀自己之前產生的統整檔
    if filename == "全部癌症統整.xlsx":
        continue

    # 不讀處理結果報告
    if filename == "處理結果.xlsx":
        continue

    excel_files.append(
        filename
    )


# 排序，讓輸出順序固定
excel_files.sort()


print(
    f"找到 {len(excel_files)} 個癌症 Excel"
)

print()


# =========================================================
# 4. 逐一讀取
# =========================================================

for filename in excel_files:

    file_path = os.path.join(
        input_folder,
        filename
    )


    print(
        f"正在讀取：{filename}"
    )


    try:

        # =================================================
        # 先取得工作表名稱
        # =================================================

        excel_file = pd.ExcelFile(
            file_path
        )

        sheet_names = (
            excel_file.sheet_names
        )


        # =================================================
        # 年齡
        # =================================================

        if "年齡" in sheet_names:

            age_df = pd.read_excel(
                file_path,
                sheet_name="年齡"
            )

            # 完全不修改內容
            if not age_df.empty:

                age_dataframes.append(
                    age_df
                )

                print(
                    f"  年齡：{len(age_df)} 筆"
                )

            else:

                print(
                    "  年齡：空白"
                )


        else:

            print(
                "  找不到「年齡」工作表"
            )


        # =================================================
        # 整併期別
        # =================================================

        if "整併期別" in sheet_names:

            stage_df = pd.read_excel(
                file_path,
                sheet_name="整併期別"
            )

            # 完全不修改內容
            if not stage_df.empty:

                stage_dataframes.append(
                    stage_df
                )

                print(
                    f"  整併期別："
                    f"{len(stage_df)} 筆"
                )

            else:

                print(
                    "  整併期別：空白"
                )


        else:

            print(
                "  找不到「整併期別」工作表"
            )


    except Exception as error:

        print(
            f"  讀取失敗：{error}"
        )


    print()


# =========================================================
# 5. 合併所有「年齡」
# =========================================================

if age_dataframes:

    combined_age = pd.concat(
        age_dataframes,
        ignore_index=True
    )

else:

    combined_age = pd.DataFrame(
        columns=[
            "年度",
            "癌別",
            "年齡",
            "合計",
            "男性",
            "女性",
            "醫學中心",
            "非醫學中心"
        ]
    )


# =========================================================
# 6. 合併所有「整併期別」
# =========================================================

if stage_dataframes:

    combined_stage = pd.concat(
        stage_dataframes,
        ignore_index=True
    )

else:

    combined_stage = pd.DataFrame(
        columns=[
            "年度",
            "癌別",
            "整併期別",
            "合計",
            "男性",
            "女性",
            "醫學中心",
            "非醫學中心"
        ]
    )


# =========================================================
# 7. Excel 欄寬設定
# =========================================================

def adjust_sheet(
    writer,
    sheet_name
):

    worksheet = writer.book[
        sheet_name
    ]


    # 凍結第一列
    worksheet.freeze_panes = "A2"


    # 自動調整欄寬
    for column_cells in worksheet.columns:

        column_letter = (
            get_column_letter(
                column_cells[0].column
            )
        )

        max_length = 0


        for cell in column_cells:

            if cell.value is None:

                value = ""

            else:

                value = str(
                    cell.value
                )


            max_length = max(
                max_length,
                len(value)
            )


        worksheet.column_dimensions[
            column_letter
        ].width = min(
            max_length + 3,
            25
        )


    # 儲存格置中
    for row in worksheet.iter_rows():

        for cell in row:

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center"
            )


# =========================================================
# 8. 輸出統整 Excel
# =========================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # 工作表 1
    combined_age.to_excel(
        writer,
        sheet_name="年齡",
        index=False
    )


    # 工作表 2
    combined_stage.to_excel(
        writer,
        sheet_name="整併期別",
        index=False
    )


    adjust_sheet(
        writer,
        "年齡"
    )


    adjust_sheet(
        writer,
        "整併期別"
    )


# =========================================================
# 9. 完成
# =========================================================

print(
    "=" * 60
)

print(
    "全部癌症統整完成"
)

print(
    "=" * 60
)

print(
    f"癌症 Excel 數量："
    f"{len(excel_files)}"
)

print(
    f"年齡總筆數："
    f"{len(combined_age)}"
)

print(
    f"整併期別總筆數："
    f"{len(combined_stage)}"
)

print()

print(
    "輸出檔案："
)

print(
    output_file
)

找到 24 個癌症 Excel

正在讀取：下咽癌.xlsx
  年齡：18 筆
  整併期別：7 筆

正在讀取：主唾液腺癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：乳癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：卵巢癌.xlsx
  年齡：18 筆
  整併期別：5 筆

正在讀取：口咽癌_1.xlsx
  年齡：18 筆
  整併期別：5 筆

正在讀取：口咽癌_2.xlsx
  年齡：18 筆
  整併期別：7 筆

正在讀取：口腔癌.xlsx
  年齡：18 筆
  整併期別：7 筆

正在讀取：喉癌.xlsx
  年齡：18 筆
  整併期別：7 筆

正在讀取：子宮頸癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：子宮體癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：惡性淋巴瘤.xlsx
  年齡：18 筆
  整併期別：5 筆

正在讀取：攝護腺癌.xlsx
  年齡：18 筆
  整併期別：5 筆

正在讀取：直腸癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：結腸癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：肝內膽管癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：肝癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：肺癌_1.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：肺癌_2.xlsx
  年齡：18 筆
  整併期別：5 筆

正在讀取：胃癌_1.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：胃癌_2.xlsx
  年齡：18 筆
  整併期別：5 筆

正在讀取：胰臟癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：膀胱癌.xlsx
  年齡：18 筆
  整併期別：5 筆

正在讀取：食道癌.xlsx
  年齡：18 筆
  整併期別：6 筆

正在讀取：鼻咽癌.xlsx
  年齡：18 筆
  整併期別：6 筆



PermissionError: [Errno 13] Permission denied: 'C:\\test2\\篩選結果\\全部癌症統整.xlsx'